In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
import torch
import pandas as pd

sys.path.append(os.getcwd())

from src.config import MODEL_NAME, DEVICE, MAX_LENGTH, LR, TRAIN_STEPS, BATCH_SIZE, SEED, ALPHA, M_IN, M_OUT
from src.utils import set_seed
from src.data import load_rows_from_csv, split_examples_by_dataset, validate_required_splits, print_dataset_counts, build_dataloaders
from src.model import load_frozen_lm, build_energy_model
from src.training import train_model
from src.evaluation import evaluate_loader, print_metrics
from src.inference import score_claim

set_seed(SEED)
print("Device:", DEVICE)



ImportError: cannot import name 'ALPHA' from 'src.config' (/home/suns11/working_dir/ebm_hallu/src/config.py)

In [5]:
CSV_PATH = "inputs/processed_qa_hallucination_dataset.csv"

In [6]:
tokenizer, base_model = load_frozen_lm(MODEL_NAME, DEVICE)
print("Loaded:", MODEL_NAME)



Loading weights: 100%|██████████████████████████████████████████████████████████████| 434/434 [00:00<00:00, 9181.84it/s]


Loaded: Qwen/Qwen2.5-3B-Instruct


In [7]:
rows = load_rows_from_csv(CSV_PATH)
splits = split_examples_by_dataset(rows)
validate_required_splits(splits)

print(f"Total paired rows: {len(splits['rows'])}")
print(f"Total individual examples: {len(splits['examples'])}")
print(f"Train HotpotQA paired rows: {len(splits['hotpot_rows'])}")
print(f"Eval HotpotQA examples: {len(splits['hotpot_examples'])}")
print(f"Eval TriviaQA examples: {len(splits['trivia_examples'])}")
print(f"Eval TruthfulQA examples: {len(splits['truthfulqa_examples'])}")
print_dataset_counts(splits["rows"], splits["examples"])



Total paired rows: 22625
Total individual examples: 45250
Train HotpotQA paired rows: 9508
Eval HotpotQA examples: 19016
Eval TriviaQA examples: 19496
Eval TruthfulQA examples: 6738

Dataset counts:
  hotpotqa: rows=9508, individual=19016, positive=9508, negative=9508
  triviaqa_wiki: rows=9748, individual=19496, positive=9748, negative=9748
  truthfulqa: rows=3369, individual=6738, positive=3369, negative=3369


In [6]:
loaders = build_dataloaders(
    splits=splits,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    use_short_answer=False,
    num_workers=0,
)



In [7]:
energy_model = build_energy_model(base_model, DEVICE, proj_dim=128, dropout=0.1)
optimizer = torch.optim.AdamW(energy_model.parameters(), lr=LR, weight_decay=1e-4)



In [8]:
history = train_model(
    loaders=loaders,
    base_model=base_model,
    energy_model=energy_model,
    optimizer=optimizer,
    device=DEVICE,
    train_steps=TRAIN_STEPS,
    alpha=ALPHA,
    m_in=M_IN,
    m_out=M_OUT,
    best_ckpt_path="best_hotpot_bce_energybound_icr_to_trivia_truthfulqa.pt",
)

pd.DataFrame(history).tail()



Epoch 000 | TotalLoss=1.3538 | BCE=0.6939 | Bound=2.0137 | HotpotAUC=0.5055 | TriviaAUC=0.5098 | TruthfulQAAUC=0.5514 | MeanEvalAUC=0.5306 | saved best
Epoch 001 | TotalLoss=1.3511 | BCE=0.6937 | Bound=2.0085 | HotpotAUC=0.5050 | TriviaAUC=0.5119 | TruthfulQAAUC=0.5617 | MeanEvalAUC=0.5368 | saved best
Epoch 002 | TotalLoss=1.3458 | BCE=0.6925 | Bound=1.9992 | HotpotAUC=0.5664 | TriviaAUC=0.5887 | TruthfulQAAUC=0.5411 | MeanEvalAUC=0.5649 | saved best
Epoch 003 | TotalLoss=1.3362 | BCE=0.6903 | Bound=1.9822 | HotpotAUC=0.6339 | TriviaAUC=0.6503 | TruthfulQAAUC=0.5425 | MeanEvalAUC=0.5964 | saved best
Epoch 004 | TotalLoss=1.2749 | BCE=0.6747 | Bound=1.8751 | HotpotAUC=0.7525 | TriviaAUC=0.7598 | TruthfulQAAUC=0.5716 | MeanEvalAUC=0.6657 | saved best
Epoch 005 | TotalLoss=1.1244 | BCE=0.6322 | Bound=1.6167 | HotpotAUC=0.8474 | TriviaAUC=0.8278 | TruthfulQAAUC=0.5867 | MeanEvalAUC=0.7072 | saved best
Epoch 006 | TotalLoss=0.9397 | BCE=0.5786 | Bound=1.3007 | HotpotAUC=0.8980 | TriviaAUC=

,epoch,total_loss,bce_loss,energy_bound_loss,in_energy_loss,out_energy_loss,hotpot_acc,hotpot_auc,hotpot_layer_auc,trivia_acc,trivia_auc,trivia_layer_auc,truthfulqa_acc,truthfulqa_auc,truthfulqa_layer_auc,mean_eval_auc,saved_best
25,25,0.315592,0.205590,0.425593,0.228408,0.197185,0.945917,0.991449,0.388085,0.772396,0.848627,0.388029,0.577619,0.595894,0.380524,0.722260,False
26,26,0.281062,0.205909,0.356214,0.156873,0.199341,0.946978,0.993931,0.389530,0.761458,0.842561,0.392463,0.568418,0.591219,0.372613,0.716890,False
27,27,0.269494,0.191712,0.347276,0.166962,0.180315,0.946448,0.991442,0.387186,0.766146,0.845135,0.385476,0.571238,0.593933,0.373881,0.719534,False
28,28,0.295032,0.195350,0.394714,0.213391,0.181323,0.945387,0.992811,0.400621,0.768229,0.834901,0.407260,0.572870,0.595502,0.370442,0.715202,False
29,29,0.242261,0.173552,0.310969,0.168898,0.142071,0.949099,0.993212,0.388097,0.766667,0.847458,0.385849,0.560701,0.590481,0.372892,0.718970,False


In [9]:
history

[{'epoch': 0,
  'total_loss': 1.3538098077379632,
  'bce_loss': 0.6939066613087093,
  'energy_bound_loss': 2.0137129621313585,
  'in_energy_loss': 1.073682975667911,
  'out_energy_loss': 0.9400299824181695,
  'hotpot_acc': 0.5,
  'hotpot_auc': 0.5054523536152415,
  'hotpot_layer_auc': 0.5005594608484238,
  'trivia_acc': 0.5,
  'trivia_auc': 0.509787868923611,
  'trivia_layer_auc': 0.4992502170138889,
  'truthfulqa_acc': 0.5,
  'truthfulqa_auc': 0.5514130592508776,
  'truthfulqa_layer_auc': 0.5375720221061182,
  'mean_eval_auc': np.float64(0.5306004640872444),
  'saved_best': True},
 {'epoch': 1,
  'total_loss': 1.3510843129830548,
  'bce_loss': 0.6937070203864056,
  'energy_bound_loss': 2.0084616278287335,
  'in_energy_loss': 0.9956126856525781,
  'out_energy_loss': 1.0128489543119898,
  'hotpot_acc': 0.5,
  'hotpot_auc': 0.5049716108761437,
  'hotpot_layer_auc': 0.49473207166946487,
  'trivia_acc': 0.5,
  'trivia_auc': 0.5118543836805556,
  'trivia_layer_auc': 0.4945730251736111,
  't